# 📘 Notebook 2d: Online Feature Store + Real-Time Inference

## Overview

A wafer comes off the line — you want a yield prediction in **milliseconds**, not a batch. This notebook enables online serving on the Feature View from Notebook 1 and deploys the champion model from Notebook 2 to a SPCS inference service.

### When to Consider Online Inference

- Single-record latency matters (<100 ms p99)
- Features already live in the Feature Store → no training/serving skew
- The model is in the Model Registry → one-call deployment to SPCS
- Request/response logging is useful for monitoring and retraining

### What's Covered

| Section | Topic |
|---|---|
| 1 | Setup — imports, session, constants |
| 2 | Enable online serving on the Feature View |
| 3 | Read features from the online store + latency benchmark |
| 4 | Create a SPCS inference service from the registered model |
| 5 | End-to-end prediction path (online FS → endpoint) |
| 6 | View captured inference data |
| 7 | Summary |

### Handoff from Notebook 1

Requires the Feature View `WAFER_YIELD_FEATURES v1.0` to be a **managed** FV with `refresh_freq` set (done in Notebook 1 after the best-practice fixes). Online serving cannot be enabled on an unmanaged FV.

### Handoff from Notebook 2

Loads the registered model `WAFER_YIELD_DEMO.ML_MODELS.WAFER_YIELD_CHAMPION v1`. If Notebook 2 registers under a different name, update the constants cell below.


## 📘 Section 1 — Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import time
import json
import requests
import numpy as np
import pandas as pd

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

from snowflake.ml.registry import Registry
from snowflake.ml.feature_store import FeatureStore, CreationMode, OnlineConfig
from snowflake.ml.feature_store.feature_view import StoreType

print("Imports OK")


In [ ]:
# ============================================================================
# SESSION + CONSTANTS
# ============================================================================
session = get_active_session()
session.sql("USE WAREHOUSE WAFER_DEMO_WH").collect()

# Feature Store (now a MANAGED FV after Notebook 01 fix - required for online serving)
FS_DATABASE      = "WAFER_YIELD_DEMO"
FS_SCHEMA        = "FEATURES"
FV_NAME          = "WAFER_YIELD_FEATURES"
FV_VERSION       = "1.0"

# Model handoff contract (must match what Notebook 02 registers).
# If you change these in 02, update them here too.
MODEL_DATABASE   = "WAFER_YIELD_DEMO"
MODEL_SCHEMA     = "ML_MODELS"
MODEL_NAME       = "WAFER_YIELD_CHAMPION"     # set by Notebook 02
MODEL_VERSION    = "v1"                        # set by Notebook 02

# SPCS service
SERVICE_DATABASE = "WAFER_YIELD_DEMO"
SERVICE_SCHEMA   = "INFERENCE"
SERVICE_NAME     = "WAFER_YIELD_REALTIME"
SERVICE_POOL     = "WAFER_REALTIME_POOL"
MIN_INSTANCES    = 1
MAX_INSTANCES    = 2

print(f"Session OK | WH: {session.get_current_warehouse()}")
print(f"Target model:   {MODEL_DATABASE}.{MODEL_SCHEMA}.{MODEL_NAME} v{MODEL_VERSION}")
print(f"Target service: {SERVICE_DATABASE}.{SERVICE_SCHEMA}.{SERVICE_NAME}")


## 📘 Section 2 — Enable Online Serving on the Feature View

Online serving is toggled per feature view via `OnlineConfig`. Behind the scenes Snowflake provisions a **Hybrid Table** for the online store and keeps it in sync with the offline Dynamic Table — no code duplication, no training/serving skew.

`target_lag="15s"` = the online store will be at most 15 seconds behind the source. Good for real-time; tighten only if fraud-detection-level freshness is required.


In [ ]:
# ============================================================================
# CONNECT TO FEATURE STORE
# ============================================================================
fs = FeatureStore(
    session=session,
    database=FS_DATABASE,
    name=FS_SCHEMA,
    default_warehouse="WAFER_DEMO_WH",
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

fv = fs.get_feature_view(FV_NAME, FV_VERSION)
print(f"Feature view: {FV_NAME} v{FV_VERSION}")
print(f"Online currently enabled: {fv.online}")


In [ ]:
# ============================================================================
# ENABLE ONLINE SERVING (idempotent — skip if already on)
# ============================================================================
if not fv.online:
    print("Enabling online serving...")
    fs.update_feature_view(
        name=FV_NAME,
        version=FV_VERSION,
        online_config=OnlineConfig(enable=True, target_lag="15s"),
    )
    # Poll until the online table is populated (initial refresh can take 30s-few min)
    t0 = time.time()
    while True:
        try:
            sample_key = session.sql(
                f"SELECT WAFER_ID FROM {FS_DATABASE}.RAW_DATA.FINAL_YIELD_LABELS LIMIT 1"
            ).collect()[0][0]
            probe = fs.read_feature_view(
                FV_NAME, FV_VERSION,
                keys=[[sample_key]],
                store_type=StoreType.ONLINE,
            )
            if probe.count() > 0:
                print(f"Online store ready in {time.time()-t0:.1f}s")
                break
        except Exception as e:
            print(f"  ...provisioning ({time.time()-t0:.0f}s): {type(e).__name__}")
        time.sleep(10)
else:
    print("Online serving already enabled — skipping")


## 📘 Section 3 — Read Features from the Online Store

Key API: `fs.read_feature_view(..., store_type=StoreType.ONLINE, keys=[[wafer_id]])` — key-based lookup on the hybrid table. Expected latency (inside Snowflake, warm cache):

| Percentile | Latency |
|---|---|
| p50 | ~30ms |
| p95 | ~50ms |
| p99 | <100ms |

First call after provisioning will be slower while the cache warms.


In [ ]:
# ============================================================================
# SINGLE-WAFER ONLINE LOOKUP
# ============================================================================
sample_wafer = session.sql(
    f"SELECT WAFER_ID FROM {FS_DATABASE}.RAW_DATA.FINAL_YIELD_LABELS LIMIT 1"
).collect()[0][0]

t0 = time.time()
features = fs.read_feature_view(
    FV_NAME, FV_VERSION,
    keys=[[sample_wafer]],
    store_type=StoreType.ONLINE,
).to_pandas()
lookup_ms = (time.time() - t0) * 1000

print(f"Wafer: {sample_wafer}")
print(f"Online lookup: {lookup_ms:.1f} ms  |  {len(features.columns)} columns")
features


In [ ]:
# ============================================================================
# BENCHMARK — 50 sequential lookups (warm cache)
# ============================================================================
wafer_ids = [r[0] for r in session.sql(
    f"SELECT WAFER_ID FROM {FS_DATABASE}.RAW_DATA.FINAL_YIELD_LABELS LIMIT 50"
).collect()]

# Warm-up
for w in wafer_ids[:5]:
    fs.read_feature_view(FV_NAME, FV_VERSION, keys=[[w]], store_type=StoreType.ONLINE).to_pandas()

# Measure
latencies_ms = []
for w in wafer_ids:
    t0 = time.time()
    fs.read_feature_view(FV_NAME, FV_VERSION, keys=[[w]], store_type=StoreType.ONLINE).to_pandas()
    latencies_ms.append((time.time() - t0) * 1000)

arr = np.array(latencies_ms)
print(f"Online lookup over {len(arr)} requests:")
print(f"   p50: {np.percentile(arr, 50):6.1f} ms")
print(f"   p95: {np.percentile(arr, 95):6.1f} ms")
print(f"   p99: {np.percentile(arr, 99):6.1f} ms")
print(f"   max: {arr.max():6.1f} ms")


## 📘 Section 4 — Create a SPCS Inference Service

`mv.create_service()` deploys the registered model to a SPCS container on a compute pool and exposes a REST endpoint. Flags:

- `service_compute_pool` — CPU or GPU pool (CPU is usually fine for single-record DNN inference)
- `ingress_enabled=True` — creates the HTTPS endpoint
- `max_instances` — horizontal scaling for concurrency
- `autocapture=True` — log every request/response to an inference table (great for monitoring & retraining)

> Takes ~5–15 min to reach `RUNNING`. The poll loop below replaces `block=True` with visible progress.


In [ ]:
# ============================================================================
# LOAD THE REGISTERED MODEL
# ============================================================================
reg = Registry(session=session, database_name=MODEL_DATABASE, schema_name=MODEL_SCHEMA)
mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)

print(f"Model: {MODEL_NAME} v{MODEL_VERSION}")
print(mv.show_functions())


In [ ]:
# ============================================================================
# CREATE SPCS SERVICE
# ============================================================================
session.sql(f"CREATE SCHEMA IF NOT EXISTS {SERVICE_DATABASE}.{SERVICE_SCHEMA}").collect()

# Skip if already running
existing = session.sql(
    f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {SERVICE_DATABASE}.{SERVICE_SCHEMA}"
).collect()

if existing:
    print(f"Service {SERVICE_NAME} already exists — reusing")
else:
    print(f"Creating service {SERVICE_NAME} on pool {SERVICE_POOL}...")
    mv.create_service(
        service_name=f"{SERVICE_DATABASE}.{SERVICE_SCHEMA}.{SERVICE_NAME}",
        service_compute_pool=SERVICE_POOL,
        ingress_enabled=True,
        min_instances=MIN_INSTANCES,
            max_instances=MAX_INSTANCES,
        autocapture=True,
    )
    print("Service request submitted")


In [ ]:
# ============================================================================
# POLL UNTIL SERVICE IS RUNNING
# ============================================================================
t0 = time.time()
while True:
    status_row = session.sql(
        f"SELECT SYSTEM$GET_SERVICE_STATUS('{SERVICE_DATABASE}.{SERVICE_SCHEMA}.{SERVICE_NAME}')"
    ).collect()[0][0]
    state = json.loads(status_row)
    all_ready = all(c.get("status") == "READY" for c in state)
    any_failed = any(c.get("status") in ("FAILED", "TERMINATING") for c in state)
    elapsed = time.time() - t0
    ready_count = sum(1 for c in state if c.get("status") == "READY")
    print(f"  [{elapsed:6.1f}s] instances READY: {ready_count}/{len(state)}")
    if all_ready:
        print(f"Service RUNNING after {elapsed/60:.1f} min")
        break
    if any_failed:
        raise RuntimeError(f"Service failed: {state}")
    time.sleep(30)

# Get endpoint
endpoints = session.sql(
    f"SHOW ENDPOINTS IN SERVICE {SERVICE_DATABASE}.{SERVICE_SCHEMA}.{SERVICE_NAME}"
).collect()
for e in endpoints:
    print(f"   {e['name']}: {e['ingress_url']}")


## 📘 Section 5 — End-to-End Prediction Path

Production flow per request:

1. Client sends a **WAFER_ID**
2. Look up that wafer's features from the online feature store
3. POST the feature vector to the SPCS endpoint
4. Return the yield prediction

Two invocation styles below:

- **Python SDK**: `mv.run(df, service_name=...)` — simplest, uses your current session
- **REST (internal endpoint)**: from a Snowsight notebook, no auth required — this is what an in-Snowflake app (Streamlit, stored proc) would use


In [ ]:
# ============================================================================
# SDK PATH — lookup features + run model in one flow
# ============================================================================
def predict_one_sdk(wafer_id: str):
    # Step 1: online feature lookup
    t0 = time.time()
    features_df = fs.read_feature_view(
        FV_NAME, FV_VERSION,
        keys=[[wafer_id]],
        store_type=StoreType.ONLINE,
    ).to_pandas()
    t_fs = (time.time() - t0) * 1000

    # Drop the join key before scoring
    X = features_df.drop(columns=["WAFER_ID"], errors="ignore").astype(np.float32)

    # Step 2: call the service
    t0 = time.time()
    preds = mv.run(X, service_name=f"{SERVICE_DATABASE}.{SERVICE_SCHEMA}.{SERVICE_NAME}")
    t_infer = (time.time() - t0) * 1000

    return {
        "wafer_id": wafer_id,
        "fs_ms": round(t_fs, 1),
        "infer_ms": round(t_infer, 1),
        "total_ms": round(t_fs + t_infer, 1),
        "prediction": preds.to_pandas().iloc[0].to_dict(),
    }

# Test on 3 wafers
for wid in wafer_ids[:3]:
    print(predict_one_sdk(wid))


In [ ]:
# ============================================================================
# REST PATH — internal endpoint (no auth needed from Snowsight notebook)
# ============================================================================
services_df = mv.list_services()
row = services_df[services_df["name"].str.upper() == SERVICE_NAME.upper()].iloc[0]
internal_url = row["internal_endpoint"]
print(f"Internal endpoint: {internal_url}")

# Assemble one payload in the External Functions format: [row_idx, col1, col2, ...]
wid = wafer_ids[0]
feat = fs.read_feature_view(
    FV_NAME, FV_VERSION,
    keys=[[wid]],
    store_type=StoreType.ONLINE,
).to_pandas().drop(columns=["WAFER_ID"], errors="ignore")

payload = {"data": [[0] + feat.iloc[0].astype(float).tolist()]}

t0 = time.time()
resp = requests.post(f"http://{internal_url}/predict", json=payload, timeout=5)
rest_ms = (time.time() - t0) * 1000

print(f"Wafer: {wid}  |  REST latency: {rest_ms:.1f} ms  |  status: {resp.status_code}")
print(f"Response: {resp.json()}")


## 📘 Section 6 — View Captured Inference Data

With `autocapture=True`, every request/response is written to an inference table. This gives you:

- Free production logs for debugging unexpected predictions
- A dataset for retraining or A/B testing
- Per-request latency + payload history


In [ ]:
# ============================================================================
# QUERY THE INFERENCE TABLE
# ============================================================================
# The inference table is discoverable via the model version
inference_df = session.sql(f'''
    SELECT *
    FROM TABLE(INFERENCE_TABLE(
        MODEL_NAME => '{MODEL_DATABASE}.{MODEL_SCHEMA}.{MODEL_NAME}',
        VERSION_NAME => '{MODEL_VERSION}'
    ))
    ORDER BY TIMESTAMP DESC
    LIMIT 20
''')

inference_df.show(10)
print(f"Rows captured: {inference_df.count():,}")


## 📘 Summary

### What We Covered

| Topic | Snowflake API |
|---|---|
| **Online feature serving** | `OnlineConfig(enable=True, target_lag="15s")` (Hybrid Table) |
| **Low-latency feature read** | `fs.read_feature_view(store_type=StoreType.ONLINE, keys=...)` |
| **Real-time model endpoint** | `mv.create_service(service_compute_pool=..., ingress_enabled=True)` |
| **Invocation (Python)** | `mv.run(df, service_name=...)` |
| **Invocation (REST)** | POST to `internal_endpoint/<function>` — no auth from Snowsight |
| **Inference logging** | `autocapture=True` → `INFERENCE_TABLE(...)` |


### Key Takeaway

Snowflake keeps the offline and online feature stores in sync automatically — the same feature definitions power training *and* serving. Combined with `mv.create_service()`, you get a production REST endpoint backed by your governed, versioned model in two calls.
